In [ ]:
!pip install -q grad-cam==1.5.7

from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/xai-brain-tumor'

import os, sys, random
if not os.path.exists('/content/xai-brain-tumor'):
    !git clone https://github.com/otaviotex/xai-brain-tumor.git
else:
    !git -C /content/xai-brain-tumor pull
sys.path.append('/content/xai-brain-tumor')

import numpy as np, pandas as pd, torch, torch.nn as nn
import matplotlib.pyplot as plt
from torchvision.models import resnet18, ResNet18_Weights

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

MET_DIR = f'{BASE}/results/metrics'
nomes = {0: 'meningioma', 1: 'glioma', 2: 'pituitary'}

def build_model():
    m = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, 3)
    return m.to(device)

from src.dataset import load_cache, make_loaders
cache = load_cache(f'{BASE}/data/cache_224.npz')
_, _, test_ds, _, _, test_dl = make_loaders(cache, f'{BASE}/data/splits', seed=SEED)
print('teste:', len(test_ds))

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

GC_DIR = f'{BASE}/results/gradcam'
os.makedirs(GC_DIR, exist_ok=True)

ckpt = torch.load(f'{BASE}/models/resnet18_B_best.pt', map_location=device)
model = build_model(); model.load_state_dict(ckpt['model']); model.eval()
cam = GradCAM(model=model, target_layers=[model.layer4[-1]])

prev, verd = {}, {}
for x, y, m, ids in test_dl:
    x = x.to(device)
    with torch.no_grad():
        pred = model(x).argmax(1).cpu().numpy()
    y = y.numpy()
    maps_pred = cam(input_tensor=x, targets=[ClassifierOutputTarget(int(c)) for c in pred])
    maps_true = cam(input_tensor=x, targets=[ClassifierOutputTarget(int(c)) for c in y])
    for j, i in enumerate(ids.tolist()):
        np.save(f'{GC_DIR}/{i}_pred.npy', maps_pred[j].astype(np.float32))
        if pred[j] != y[j]:
            np.save(f'{GC_DIR}/{i}_true.npy', maps_true[j].astype(np.float32))
        prev[i] = int(pred[j]); verd[i] = int(y[j])
print('mapas salvos para', len(prev), 'imagens')

ref = pd.read_csv(f'{MET_DIR}/predicoes_teste.csv')
print('previsões idênticas às da Etapa 9:',
      all(nomes[prev[i]] == p for i, p in zip(ref.id, ref.previsto)))

In [ ]:
rng = np.random.default_rng(0)
amostra = rng.choice(sorted(prev), size=6, replace=False)
fig, axs = plt.subplots(1, 6, figsize=(18, 3.8))
for ax, i in zip(axs, amostra):
    k = cache['row_of'][int(i)]
    img = cache['X'][k]; mask = cache['M'][k].astype(bool)
    cm_ = np.load(f'{GC_DIR}/{i}_pred.npy')
    ax.imshow(img, cmap='gray'); ax.imshow(cm_, cmap='jet', alpha=0.4)
    ax.contour(mask, levels=[0.5], colors='white', linewidths=1)
    ax.set_title(f'id {i} | real: {nomes[verd[i]]}\nprevisto: {nomes[prev[i]]}', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.savefig(f'{GC_DIR}/amostra_gradcam.png', dpi=150); plt.show()

In [ ]:
from scipy.stats import wilcoxon

def metricas(cam_map, mask):
    mask = mask.astype(bool)
    out = {}
    for t in (0.3, 0.5, 0.7):
        b = cam_map >= t
        uni = np.logical_or(b, mask).sum()
        out[f'iou_{t}'] = np.logical_and(b, mask).sum() / uni if uni else np.nan
    ij = np.unravel_index(np.argmax(cam_map), cam_map.shape)
    out['pointing'] = int(mask[ij])
    total = cam_map.sum()
    dentro = cam_map[mask].sum() / total if total > 0 else np.nan
    out['razao_energia'] = dentro / mask.mean() if mask.any() else np.nan
    return out

rng = np.random.default_rng(42)
ids_teste = sorted(prev)
por_classe = {c: [i for i in ids_teste if verd[i] == c] for c in range(3)}

linhas = []
for i in ids_teste:
    k = cache['row_of'][i]
    mask = cache['M'][k].astype(bool)
    camp = np.load(f'{GC_DIR}/{i}_pred.npy')
    outro = int(rng.choice([j for j in por_classe[verd[i]] if j != i]))
    mask_c = cache['M'][cache['row_of'][outro]].astype(bool)
    r, c = metricas(camp, mask), metricas(camp, mask_c)
    linhas.append(dict(id=i, real=nomes[verd[i]], previsto=nomes[prev[i]],
                       acertou=prev[i] == verd[i], area_mascara=mask.mean(), id_controle=outro,
                       **r, **{f'ctrl_{n}': v for n, v in c.items()}))
df = pd.DataFrame(linhas)
df.to_csv(f'{GC_DIR}/metricas_xai_teste.csv', index=False)

cols = ['iou_0.3', 'iou_0.5', 'iou_0.7', 'pointing', 'razao_energia']
tab = pd.DataFrame([dict(metrica=c,
        real_media=df[c].mean(), real_mediana=df[c].median(), real_dp=df[c].std(),
        ctrl_media=df['ctrl_' + c].mean(), ctrl_mediana=df['ctrl_' + c].median(),
        ctrl_dp=df['ctrl_' + c].std()) for c in cols]).round(3)
print(f'REAL x CONTROLE (n = {len(df)})'); print(tab.to_string(index=False))
tab.to_csv(f'{GC_DIR}/tabela_xai_real_vs_controle.csv', index=False)

print('\nPOR CLASSE REAL'); print(df.groupby('real')[cols].agg(['mean', 'median']).round(3).to_string())
print('\nACERTOS x ERROS'); print(df.groupby('acertou')[cols].agg(['mean', 'median']).round(3).to_string())
for c in ['iou_0.5', 'razao_energia']:
    print(f'Wilcoxon pareado {c}: p =', wilcoxon(df[c], df['ctrl_' + c]).pvalue)

In [ ]:
le = []
for i in df[~df.acertou].id:
    k = cache['row_of'][i]
    camt = np.load(f'{GC_DIR}/{i}_true.npy')
    le.append(dict(id=i, **{f'verdadeira_{n}': v for n, v in metricas(camt, cache['M'][k].astype(bool)).items()}))
df_e = pd.DataFrame(le)
df_e.to_csv(f'{GC_DIR}/metricas_xai_erros_classe_verdadeira.csv', index=False)
print(len(df_e), 'erros'); print(df_e.drop(columns='id').describe().round(3).to_string())

In [ ]:
print(df.groupby('real').area_mascara.agg(['mean', 'median']).round(3))
cob = [(np.load(f'{GC_DIR}/{i}_pred.npy') >= 0.5).mean() for i in df.id]
print('fração média da imagem acima de 0,5:', round(float(np.mean(cob)), 3))

In [ ]:
import json, sys, importlib.metadata as md
import torch, torchvision, numpy as np, sklearn, scipy

GC_DIR = f'{BASE}/results/gradcam'
SEED = 42

cfg = dict(
    seed=SEED,
    camada_alvo='layer4[-1]',
    classe_alvo='prevista (e verdadeira nos erros)',
    entrada='224x224',
    limiar_primario=0.5,
    limiares_sensibilidade=[0.3, 0.7],
    metricas=['IoU', 'pointing game', 'razão de energia'],
    controle='máscara de outra imagem do teste da mesma classe, rng seed 42',
    teste_estatistico='Wilcoxon pareado',
    checkpoint='resnet18_B_best.pt',
    n_teste=498,
    versoes=dict(
        python=sys.version.split()[0],
        torch=torch.__version__,
        torchvision=torchvision.__version__,
        numpy=np.__version__,
        sklearn=sklearn.__version__,
        scipy=scipy.__version__,
        grad_cam=md.version('grad-cam'),
    ),
)
json.dump(cfg, open(f'{GC_DIR}/config_gradcam.json', 'w'), indent=2, ensure_ascii=False)
print(json.dumps(cfg, indent=2, ensure_ascii=False))